In [1]:

from rdflib import Graph, Namespace, RDF, URIRef
import requests
import os
DCAT = Namespace("http://www.w3.org/ns/dcat#")
DCT  = Namespace("http://purl.org/dc/terms/")
def detect_format(source, content=None):
    """
    Detect the RDF format based on:
    - file extension
    - or the presence of JSON-LD keywords in content
    """

    # If we have content, try to detect JSON-LD
    if content is not None:
        trimmed = content.strip()
        if trimmed.startswith("{") and "@context" in trimmed:
            return "json-ld"

    # Otherwise use file extension
    ext = source.lower()

    if ext.endswith(".ttl"):
        return "turtle"
    if ext.endswith(".json") or ext.endswith(".jsonld"):
        return "json-ld"
    if ext.endswith(".rdf") or ext.endswith(".xml"):
        return "xml"
    if ext.endswith(".nt"):
        return "nt"

    # Default fallback
    return "turtle"

def load_graph(source):
    g = Graph()

    # Case 1 — URL
    if source.startswith("http://") or source.startswith("https://"):
        print(f"Fetching DCAT file from URL:\n  {source}\n")
        r = requests.get(source)
        r.raise_for_status()
        content = r.text
        fmt = detect_format(source, content)
        print(f"Detected format: {fmt}\n")
        g.parse(data=content, format=fmt)
        return g

    # Case 2 — Local file
    if not os.path.exists(source):
        raise FileNotFoundError(f"Local file not found: {source}")

    print(f"Loading DCAT file from disk:\n  {source}\n")
    with open(source, "r", encoding="utf-8") as f:
        content = f.read()

    fmt = detect_format(source, content)
    print(f"Detected format: {fmt}\n")

    g.parse(data=content, format=fmt)
    return g

def extract_catalog_structure(g):
    """
    Extracts full catalog → dataset → related classes structure from a DCAT-US 1.1 graph.
    Returns a nested dictionary suitable for comparison to DCAT-US 3.0 requirements.
    """

    catalogs = list(g.subjects(RDF.type, DCAT.Catalog))
    if not catalogs:
        raise ValueError("No dcat:Catalog found in the file!")

    result = {}

    for catalog in catalogs:
        catalog_uri = str(catalog)
        result[catalog_uri] = {"datasets": []}

        # 1. All datasets linked to this catalog
        dataset_uris = list(g.objects(catalog, DCAT.dataset))

        for ds in dataset_uris:
            ds_entry = {
                "uri": str(ds),
                "properties": {},
                "related_classes": {}
            }

            # -------------------------------------------
            # 2. Collect ALL triples describing the dataset
            # -------------------------------------------

            for p, o in g.predicate_objects(ds):
                pred = str(p)

                # add property to dataset's property list
                ds_entry["properties"].setdefault(pred, [])
                ds_entry["properties"][pred].append(str(o))

                # -------------------------------------------
                # 3. If object is a resource with a class, capture it
                # -------------------------------------------
                if isinstance(o, URIRef):
                    o_classes = list(g.objects(o, RDF.type))
                    for cls in o_classes:
                        cls_uri = str(cls)
                        ds_entry["related_classes"].setdefault(cls_uri, set())
                        ds_entry["related_classes"][cls_uri].add(str(o))

            # Convert related class sets to lists
            for k in list(ds_entry["related_classes"]):
                ds_entry["related_classes"][k] = list(ds_entry["related_classes"][k])

            # Store dataset entry
            result[catalog_uri]["datasets"].append(ds_entry)

    return result

In [2]:
g = load_graph("../test.json")

Loading DCAT file from disk:
  ../test.json

Detected format: json-ld



In [3]:
a=extract_catalog_structure(g)

In [9]:
a.keys()

dict_keys(['N2a329ca238db4a5fb89517d3abdcefce'])

In [4]:
for catl,dct in a.items():
    for ds in dct["datasets"]:
        print(ds)

{'uri': 'https://data.colorado.gov/resource/abcd-1234', 'properties': {'http://www.w3.org/1999/02/22-rdf-syntax-ns#type': ['http://www.w3.org/ns/dcat#Dataset'], 'http://purl.org/dc/terms/title': ['Active Business Licenses'], 'http://purl.org/dc/terms/description': ['All active business licenses issued by the Colorado Department of Revenue.'], 'http://www.w3.org/ns/dcat#keyword': ['business', 'licenses', 'DOR'], 'http://www.w3.org/ns/dcat#theme': ['Economy'], 'http://purl.org/dc/terms/issued': ['2020-01-15'], 'http://purl.org/dc/terms/modified': ['2024-03-10'], 'http://www.w3.org/ns/dcat#landingPage': ['https://data.colorado.gov/Business/Active-Business-Licenses/abcd-1234'], 'http://purl.org/dc/terms/publisher': ['Nde23d572f2eb4d04a25f9a539a03d4e8'], 'http://www.w3.org/ns/dcat#contactPoint': ['N150699d86d0844e58fd419c3eac9dc74'], 'http://www.w3.org/ns/dcat#distribution': ['Ne6dc92f1b26a49baa1283209c514fffe', 'N2a198f30c6ea4b2f8eb98998b73c24fa'], 'http://purl.org/dc/terms/spatial': ['N43